# Phase 4: Aggregations & Window Functions

Turn the enriched `trips_with_zones` fact table into real metrics: busiest
zones by hour, each zone's rank within its borough, and running/cumulative
totals — the kind of query that sits behind a dashboard or report.

**Docs:** [Window Functions guide](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html) | [`pyspark.sql.Window` API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html) | [`pyspark.sql.functions` (rank/row_number/lag/lead)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

## Concept: `groupBy().agg()` vs. window functions

`groupBy().agg()` **collapses** N rows into 1 row per group — e.g. total
trip count per pickup zone. You lose row-level detail; you can't get back
to "which individual trip" contributed to that total.

A **window function** computes a value *per row*, using a `Window` spec
(`partitionBy` + `orderBy`) that defines "this row's group" and "this row's
position within that group" — without collapsing rows. Same input row count
in and out. This is what you need for:
- "rank of this zone within its borough" (`F.rank()` / `F.dense_rank()`)
- "the Nth busiest hour, keeping ties" (`F.row_number()` vs `F.rank()`)
- "running total up to this row" (`F.sum(...).over(window)` with an ordered,
  unbounded-preceding frame)
- "previous/next row's value" (`F.lag()` / `F.lead()`) — e.g. time between
  consecutive pickups in the same zone

## Anatomy of a `Window` spec

```python
from pyspark.sql import Window

w = (
    Window
    .partitionBy("pickup_borough")   # "reset" the window per borough
    .orderBy(F.desc("trip_count"))    # order rows within each partition
)
```

`partitionBy` is exactly like `groupBy`'s grouping columns — but instead of
collapsing, each row keeps its full detail *plus* a computed column from the
window function. `orderBy` matters for ranking functions and for any
running/cumulative aggregate (order determines what "running" means).


In [ ]:
# Setup: same read/cast/clean/join pipeline from notebooks 01-03, now
# imported from the dataforge_ai package instead of duplicated inline.
import os
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from dataforge_ai import read_trips, clean_trips, join_zones

spark = (
    SparkSession.builder
    .appName("DataForge-Phase4-Aggregations")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

RAW = "../data/raw"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"

paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]
trips_clean = clean_trips(read_trips(spark, paths))

zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW}/taxi_zone_lookup.csv")
)

trips_with_zones = join_zones(trips_clean, zones).withColumn(
    "pickup_hour", F.hour("tpep_pickup_datetime")
)

print("trips_with_zones:", trips_with_zones.count(), "rows")
trips_with_zones.select(
    "pickup_zone", "pickup_borough", "dropoff_zone", "pickup_hour", "fare_amount"
).show(5)

trips_with_zones: 9301798 rows
+--------------------+--------------+--------------------+-----------+-----------+
|         pickup_zone|pickup_borough|        dropoff_zone|pickup_hour|fare_amount|
+--------------------+--------------+--------------------+-----------+-----------+
|   LaGuardia Airport|        Queens|    Brooklyn Heights|          0|       44.3|
|        East Village|     Manhattan|Penn Station/Madi...|          0|       10.0|
|            Flatiron|     Manhattan|        Clinton East|          0|       19.8|
| Lincoln Square West|     Manhattan|Sutton Place/Turt...|          0|       15.6|
|Upper East Side N...|     Manhattan|   East Harlem South|          0|        5.1|
+--------------------+--------------+--------------------+-----------+-----------+
only showing top 5 rows



## Worked example: rank each pickup zone within its borough

Goal: for each borough, rank its pickup zones by trip count (busiest = rank 1).
This needs a `groupBy` first (to get counts per zone) **then** a window
function over that result (to rank within each borough) — a common two-step
pattern: aggregate first, then rank the aggregated rows.


In [2]:
zone_counts = (
    trips_with_zones
    .groupBy("pickup_borough", "pickup_zone")
    .agg(F.count("*").alias("trip_count"))
)

zone_rank_window = Window.partitionBy("pickup_borough").orderBy(F.desc("trip_count"))

zone_ranked = zone_counts.withColumn(
    "borough_rank", F.rank().over(zone_rank_window)
)

# Show only the #1 zone per borough (rows aren't collapsed, so we filter).
zone_ranked.filter(F.col("borough_rank") == 1).orderBy(F.desc("trip_count")).show(truncate=False)


+--------------+---------------------------+----------+------------+
|pickup_borough|pickup_zone                |trip_count|borough_rank|
+--------------+---------------------------+----------+------------+
|Queens        |JFK Airport                |454279    |1           |
|Manhattan     |Upper East Side South      |434034    |1           |
|Unknown       |N/A                        |104178    |1           |
|N/A           |Outside of NYC             |7013      |1           |
|Brooklyn      |Downtown Brooklyn/MetroTech|5128      |1           |
|Bronx         |Mott Haven/Port Morris     |1211      |1           |
|EWR           |Newark Airport             |1188      |1           |
|Staten Island |Bloomfield/Emerson Hill    |260       |1           |
+--------------+---------------------------+----------+------------+



## Hands-on task: hourly demand profile per borough

Build `hourly_demand(trips)` that, for each `(pickup_borough, pickup_hour)`:
1. Computes `trip_count` and `avg_fare` (average `fare_amount`).
2. Ranks each hour *within its borough* by `trip_count` (busiest hour = rank 1)
   using `F.dense_rank()` (not `F.rank()` — explained below).
3. Adds a running total column `cumulative_trips`: the cumulative sum of
   `trip_count` as you go hour-by-hour (0→23) within each borough — useful
   for "by what hour has a borough seen 50% of its daily trips" type analysis.

Then use it to answer: **what hour is busiest for Manhattan, and by what hour
has Manhattan crossed 50% of its total daily trip volume?**


In [3]:
def hourly_demand(trips: DataFrame) -> DataFrame:
    """Per-borough, per-hour trip_count/avg_fare, ranked by demand within the
    borough, plus a running total of trips as the day progresses (hour 0-23).

    `F.dense_rank()` over `F.rank()`: with `rank`, ties leave gaps (two rows
    tied for #1 means the next rank is #3) -- `dense_rank` keeps ranks
    consecutive (next rank is #2). For "busiest hour" ties are unlikely with
    real trip counts, but dense_rank is the safer default when ties are
    possible and you don't want gaps confusing downstream consumers.

    The running-total window orders by `pickup_hour` (not trip_count) --
    order defines what "cumulative" means. Its frame is unbounded preceding
    to current row (Window's default when `orderBy` is set), so each row's
    `cumulative_trips` = sum of trip_count for hour 0 through this hour.
    """
    hourly = (
        trips
        .groupBy("pickup_borough", "pickup_hour")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare"),
        )
    )

    rank_window = Window.partitionBy("pickup_borough").orderBy(F.desc("trip_count"))
    running_window = Window.partitionBy("pickup_borough").orderBy("pickup_hour")

    return (
        hourly
        .withColumn("demand_rank", F.dense_rank().over(rank_window))
        .withColumn("cumulative_trips", F.sum("trip_count").over(running_window))
        .orderBy("pickup_borough", "pickup_hour")
    )


demand = hourly_demand(trips_with_zones)

# Busiest hour for Manhattan:
demand.filter((F.col("pickup_borough") == "Manhattan") & (F.col("demand_rank") == 1)).show()

# Full hourly profile + when Manhattan crosses 50% of its daily total:
manhattan = demand.filter(F.col("pickup_borough") == "Manhattan")
total_manhattan_trips = manhattan.agg(F.sum("trip_count")).first()[0]

manhattan.withColumn(
    "pct_of_daily_total", F.round(F.col("cumulative_trips") / F.lit(total_manhattan_trips) * 100, 1)
).select("pickup_hour", "trip_count", "cumulative_trips", "pct_of_daily_total").show(24)


+--------------+-----------+----------+------------------+-----------+----------------+
|pickup_borough|pickup_hour|trip_count|          avg_fare|demand_rank|cumulative_trips|
+--------------+-----------+----------+------------------+-----------+----------------+
|     Manhattan|         18|    605346|14.430608709729716|          1|         6111701|
+--------------+-----------+----------+------------------+-----------+----------------+

+-----------+----------+----------------+------------------+
|pickup_hour|trip_count|cumulative_trips|pct_of_daily_total|
+-----------+----------+----------------+------------------+
|          0|    217349|          217349|               2.6|
|          1|    155045|          372394|               4.5|
|          2|    104952|          477346|               5.8|
|          3|     71271|          548617|               6.6|
|          4|     45275|          593892|               7.2|
|          5|     42241|          636133|               7.7|
|         

## Further reading

- [Window Functions in Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html) — ranking, analytic, and aggregate window functions
- [`pyspark.sql.functions` reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) — `rank`, `dense_rank`, `row_number`, `lag`, `lead`, `sum().over(...)`
- [Spark SQL Performance Tuning Guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html) — relevant next since window functions require a shuffle per `partitionBy` (worth checking `.explain()` on `demand` to see the `Window` + `Exchange` operators)

**Note:** examples built against PySpark 3.5.3 (this container's version);
the official docs above may reference newer APIs (latest is 4.2.0 as of this
writing) — check version-specific behavior if something doesn't match.
